# Smart Fund Advisor — Notebook 2: Central Model Training (v2)

**Objective:**  
Train the Risk Appetite MLP on **70 % of unique customers** (the central server split).  
The saved weights serve as the **initial global model** for federated learning.

### Architecture (v2)
```
Input(15) → FC(256)→BN→GELU→Drop(0.3)
         → FC(128)→BN→GELU→Drop(0.3) + Residual(Input→128)
         → FC(64) →BN→GELU→Drop(0.3)
         → FC(32) →BN→GELU→Drop(0.15)
         → FC(5)
```
- **Loss**: FocalLoss (gamma=2.0, label_smoothing=0.05) — boosts tail-class recall  
- **Optimizer**: AdamW + OneCycleLR (super-convergence)  
- **Early stopping**: patience=8, gradient clipping max_norm=5.0  
- **15 features** including EMI_Income_Ratio, Savings_Rate, Credit_History_Score

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split

from config import RISK_FEATURES, CENTRAL_SPLIT, RANDOM_SEED, RISK_CLASSES
from src.preprocessing import get_clean_customer_data
from src.risk_labeling  import assign_risk_label
from src.central_model  import train_central_model, save_central_model, predict, print_classification_report
from src.utils          import set_seed, plot_training_history, plot_confusion_matrix, plot_risk_distribution

set_seed(RANDOM_SEED)
print('Setup complete ✓')

Setup complete ✓


In [2]:
# ── 1. Load, engineer, label ──
print('Loading and preprocessing data...')
df = get_clean_customer_data(fit_scaler=True)
df = assign_risk_label(df, fit_encoder=True)

print(f'Total customers: {len(df)}')
print('Risk label distribution:')
print(df['risk_label'].value_counts())

Loading and preprocessing data...


Total customers: 12500
Risk label distribution:
risk_label
Low          3125
High         3125
Medium       3124
Very_High    1563
Very_Low     1563
Name: count, dtype: int64


In [3]:
# ── 2. 70 / 30 customer-level split ──
all_customers = df['Customer_ID'].unique()
central_cust, fl_cust = train_test_split(
    all_customers, train_size=CENTRAL_SPLIT, random_state=RANDOM_SEED
)

df_central = df[df['Customer_ID'].isin(central_cust)].copy()
df_fl      = df[df['Customer_ID'].isin(fl_cust)].copy()

print(f'Central split  : {len(df_central)} customers ({len(df_central)/len(df)*100:.1f}%)')
print(f'FL split       : {len(df_fl)}      customers ({len(df_fl)/len(df)*100:.1f}%)')

# Save FL split Customer IDs for use in Notebook 03
fl_cust_df = df_fl[['Customer_ID', 'risk_label_encoded']]
fl_cust_df.to_csv('models/fl_customer_split.csv', index=False)
print('FL split saved → models/fl_customer_split.csv')

Central split  : 8125 customers (65.0%)
FL split       : 4375      customers (35.0%)
FL split saved → models/fl_customer_split.csv


In [4]:
# ── 3. Prepare X, y arrays ──
feat_cols = [f for f in RISK_FEATURES if f in df_central.columns]

X_all = df_central[feat_cols].values.astype(np.float32)
y_all = df_central['risk_label_encoded'].values.astype(np.int64)

# 80 / 20 train / val within the central split
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_SEED
)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Class distribution (train): {np.bincount(y_train)}')

Train: (6500, 15)  |  Val: (1625, 15)
Class distribution (train): [1645 1642 1601  805  807]


In [5]:
# ── 4. Risk label distribution (central split) ──
plot_risk_distribution(
    y_train,
    title='Risk Appetite — Central Training Split',
    save_path='models/plot_central_train_dist.png'
)

Saved distribution plot → models/plot_central_train_dist.png


In [6]:
# ── 5. TRAIN (v2: FocalLoss + OneCycleLR + early stopping) ──
from config import EPOCHS
print(f'Training central model (FocalLoss gamma=2.0, GELU, residual, epochs={EPOCHS})...')
model, history = train_central_model(
    X_train, y_train, X_val, y_val,
    epochs=EPOCHS, verbose=True
)
save_central_model(model, history)
print('Done ✓')

# Show model architecture
print(f'\nModel architecture:')
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

Training central model (FocalLoss gamma=2.0, GELU, residual, epochs=40)...
[Central] Training on device: cpu
[Central] Features: 15 | Focal Loss γ=2.0 | Label smoothing α=0.05 | Early stop patience=8


  Epoch   1/40  train_loss=0.0196  val_loss=0.0107  val_acc=0.4535  ★ best


  Epoch   2/40  train_loss=0.0110  val_loss=0.0066  val_acc=0.5422  ★ best


  Epoch   3/40  train_loss=0.0069  val_loss=0.0040  val_acc=0.7182  ★ best


  Epoch   4/40  train_loss=0.0047  val_loss=0.0026  val_acc=0.8289  ★ best


...

  Epoch  24/40  train_loss=0.0017  val_loss=0.0010  val_acc=0.9163  ★ best


  Epoch  25/40  train_loss=0.0017  val_loss=0.0011  val_acc=0.8745


  Epoch  30/40  train_loss=0.0017  val_loss=0.0010  val_acc=0.9255  ★ best


  Epoch  35/40  train_loss=0.0016  val_loss=0.0010  val_acc=0.9212


  ── Early stopping at epoch 38 (no improvement for 8 epochs)
[Central] Best val accuracy: 0.9255 (stopped at epoch 38/40)
[Central] Model saved → /Users/chaitanya/Downloads/Submission/Code/20Feb26/models/central_risk_model.pt
Done ✓

Model architecture:
RiskMLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): GELU(approximate='none')
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=64, out_feat

In [7]:
# ── 6. Training history ──
plot_training_history(history, save_path='models/plot_central_training.png')

Saved training history plot → models/plot_central_training.png


In [8]:
# ── 7. Validation evaluation ──
y_pred, y_probs = predict(model, X_val)
print_classification_report(y_val, y_pred)

from src.utils import print_full_report
# print_full_report now auto-loads le.classes_ (alphabetical) as default label order
print_full_report(y_val, y_pred)


Classification Report
              precision    recall  f1-score   support

        High       0.97      0.88      0.92       411
         Low       0.98      0.88      0.93       411
      Medium       0.96      0.95      0.95       400
   Very_High       0.82      1.00      0.90       201
    Very_Low       0.83      1.00      0.91       202

    accuracy                           0.93      1625
   macro avg       0.91      0.94      0.92      1625
weighted avg       0.93      0.93      0.93      1625

Confusion Matrix:
[[360   0   7  44   0]
 [  0 361   9   0  41]
 [ 13   7 380   0   0]
 [  0   0   0 201   0]
 [  0   0   0   0 202]]

 CLASSIFICATION REPORT — Risk Appetite (5 Classes)
              precision    recall  f1-score   support

        High       0.97      0.88      0.92       411
         Low       0.98      0.88      0.93       411
      Medium       0.96      0.95      0.95       400
   Very_High       0.82      1.00      0.90       201
    Very_Low       0.83      1.

In [9]:
# ── 8. Confusion matrix ──
plot_confusion_matrix(
    y_val, y_pred,
    save_path='models/plot_central_confusion.png'
)

Saved confusion matrix → models/plot_central_confusion.png


In [10]:
# ── 9. Predicted probability distribution per class ──
import matplotlib.pyplot as plt
import joblib

# le.classes_ gives alphabetical order matching MLP output columns:
# col 0=High, 1=Low, 2=Medium, 3=Very_High, 4=Very_Low
le = joblib.load('models/label_encoder.joblib')
class_names = list(le.classes_)

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for i, (ax, cls) in enumerate(zip(axes, class_names)):
    ax.hist(y_probs[:, i], bins=30, color=f'C{i}', edgecolor='white', alpha=0.8)
    ax.set_title(f'P({cls})')
    ax.set_xlabel('Probability')
axes[0].set_ylabel('Count')
plt.suptitle('Predicted Probability Distributions (Validation Set)', fontsize=13)
plt.tight_layout()
plt.savefig('models/plot_central_prob_dist.png', dpi=150)
plt.show()

In [11]:
# ── 10. Save FL-split DataFrame for Notebook 03 ──
# We need the full feature + label rows for FL clients
df_fl.to_csv('models/df_fl_split.csv', index=False)
print('FL DataFrame saved → models/df_fl_split.csv')
print('\nCentral model training COMPLETE.')
print('  Saved: models/central_risk_model.pt')
print('  Saved: models/feature_scaler.joblib')
print('  Saved: models/label_encoder.joblib')

FL DataFrame saved → models/df_fl_split.csv

Central model training COMPLETE.
  Saved: models/central_risk_model.pt
  Saved: models/feature_scaler.joblib
  Saved: models/label_encoder.joblib


---
## Key Results (v2)
| Metric | Value |
|--------|-------|
| Architecture | MLP 15→256→128→64→32→5 (GELU, residual, BatchNorm) |
| Loss function | FocalLoss (gamma=2.0, label_smoothing=0.05) |
| Scheduler | OneCycleLR (super-convergence) |
| Parameters | 50,501 |
| Training customers | 70% of total (8,750) |
| Early stopping | patience=8 |
| Gradient clipping | max_norm=5.0 |

→ Proceed to **Notebook 03** for Federated Learning simulation.